# Predicción de costos de seguro (insurance.csv)

En este notebook vamos a construir paso a paso un pipeline para predecir la columna `charges` (costo del seguro) usando modelos de regresión.

## 1. Importar librerías y cargar datos

En este paso importamos las librerías principales y cargamos el archivo `insurance.csv` en un DataFrame de pandas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Cargar el dataset
df = pd.read_csv('insurance.csv')
df.head()

## 2. EDA rápido (exploración de datos)

Aquí revisamos información básica del dataset: dimensiones, tipos de datos, valores faltantes y estadísticas descriptivas de las variables numéricas.

In [ ]:
# 2.1 Forma del dataset
print('Shape:', df.shape)

# 2.2 Tipos de datos
print('
Tipos de datos:')
print(df.dtypes)

# 2.3 Valores faltantes
print('
Valores faltantes por columna:')
print(df.isna().sum())

# 2.4 Estadísticas de variables numéricas
df.describe()

### 2.5 Distribución de la variable objetivo `charges`

Visualizamos la distribución de `charges` para entender su forma (asimetría, posibles outliers).

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['charges'], kde=True)
plt.title('Distribución de charges')
plt.show()

### 2.6 Correlación numérica con `charges`

Calculamos la correlación de las variables numéricas con `charges` para ver qué tan lineal es su relación.

In [ ]:
corr = df.corr(numeric_only=True)
corr['charges'].sort_values(ascending=False)

## 3. Preparación de datos (encoding y train/test split)

Definimos la variable objetivo `y` y las variables explicativas `X`.
Luego identificamos qué columnas son numéricas y cuáles categóricas, para aplicar `OneHotEncoder` a las categóricas dentro de un `ColumnTransformer`.

In [ ]:
# 3.1 Definir X e y
X = df.drop('charges', axis=1)
y = df['charges']

# 3.2 Columnas numéricas y categóricas
numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

# 3.3 Preprocesador: OneHotEncoder para categóricas, 'passthrough' para numéricas
numeric_transformer = 'passthrough'
categorical_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
 )

# 3.4 Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape

## 4. Modelo 1: Regresión Lineal

Primero entrenamos un modelo de **Regresión Lineal múltiple** como línea base.
Armamos un `Pipeline` que incluye el preprocesamiento y el modelo.

In [ ]:
# 4.1 Definir el pipeline de regresión lineal
linreg_model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LinearRegression())
])

# 4.2 Entrenar
linreg_model.fit(X_train, y_train)

# 4.3 Predicciones y métricas en test
y_pred_lin = linreg_model.predict(X_test)
rmse_lin = mean_squared_error(y_test, y_pred_lin, squared=False)
r2_lin = r2_score(y_test, y_pred_lin)

print('Regresión Lineal - RMSE:', rmse_lin)
print('Regresión Lineal - R2:', r2_lin)

## 5. Modelo 2: Random Forest Regressor

Ahora entrenamos un **RandomForestRegressor**, que suele capturar relaciones no lineales y efectos de interacción.

In [ ]:
# 5.1 Definir el pipeline de Random Forest
rf_model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# 5.2 Entrenar
rf_model.fit(X_train, y_train)

# 5.3 Predicciones y métricas en test
y_pred_rf = rf_model.predict(X_test)
rmse_rf = mean_squared_error(y_test, y_pred_rf, squared=False)
r2_rf = r2_score(y_test, y_pred_rf)

print('Random Forest - RMSE:', rmse_rf)
print('Random Forest - R2:', r2_rf)

## 6. Comparación de modelos

En este paso comparamos las métricas principales (RMSE y R²) de cada modelo en el conjunto de test.

In [ ]:
results = pd.DataFrame({
    'modelo': ['Regresión Lineal', 'Random Forest'],
    'RMSE_test': [rmse_lin, rmse_rf],
    'R2_test': [r2_lin, r2_rf]
})
results